# EDSS DuckDB validation

## tl;dr

The restricted DuckDB contains all 233 cataloged EDSS panels and 180,119,183 source rows without flattening incompatible grains. The employment analysis layer enforces the schema break: the legacy OpenID view contains 7,277,987 rows from 2010–2022 only, while the standalone 2023–2024 view contains 46,962 rows and exposes neither canonical nor candidate OpenID.

## Context & Methods

This notebook opens the generated database read-only and compares its manifest, information schema, and table dimensions with the committed panel catalog.

### Key Assumptions

- `edss_panel_catalog.csv` is the authoritative list of logical panels and row counts.
- Catalog `column_count` measures domain columns; each panel CSV also has 12 provenance columns.
- Blank strings are meaningful source values and are not converted to SQL `NULL`.
- The database is restricted because the historical employment table may contain sensitive fields.
- The committed identity-resolution summary is the authority for excluding all 2023–2024 employment rows from legacy OpenID longitudinal analysis.

In [1]:
from pathlib import Path
import csv
import json
import duckdb

repo_root = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
database_path = repo_root / 'data/processed/edss/restricted/edss_all.duckdb'
catalog_path = repo_root / 'data/metadata/edss_panel_catalog.csv'
audit_path = repo_root / 'data/metadata/edss_duckdb_build.json'
catalog = list(csv.DictReader(catalog_path.open(encoding='utf-8-sig', newline='')))
audit = json.loads(audit_path.read_text(encoding='utf-8'))
con = duckdb.connect(str(database_path), read_only=True)
print({'duckdb_version': duckdb.__version__, 'database_bytes': database_path.stat().st_size, 'audit_sha256': audit['database']['sha256']})

{'duckdb_version': '1.4.1', 'database_bytes': 16048992256, 'audit_sha256': 'a7c68f21f624eda07230bdf4ce9b5ef74bb9532ece76b770da1bb1c53f2db3de'}


## Data

The manifest and catalog expose table-level source, dimensions, and checksums without scanning unrelated grains together.

In [2]:
source_summary = con.execute("""
    SELECT source, count(*) AS panel_tables, sum(loaded_rows) AS panel_rows
    FROM meta.load_manifest
    GROUP BY source
    ORDER BY source
""").fetchall()
database_summary = con.execute('SELECT * FROM meta.database_summary').fetchone()
print('source_summary')
for row in source_summary:
    print(row)
print('database_summary', database_summary)

source_summary
('고등교육통계', 102, 147761413)
('대학정보공시', 130, 25032821)
('취업통계', 1, 7324949)
database_summary ('complete', 233, 180119183, 7277987, 46962, 46962)


## Results

Each physical DuckDB table is checked against the catalog row count and the domain-plus-provenance column count.

In [3]:
schema_for_source = {
    '고등교육통계': 'higher_education',
    '대학정보공시': 'university_disclosure',
    '취업통계': 'employment',
}
dimension_mismatches = []
for item in catalog:
    schema = schema_for_source[item['source']]
    table = 'panel_' + item['catalog_code'].lower()
    actual_rows = con.execute(f'SELECT count(*) FROM "{schema}"."{table}"').fetchone()[0]
    actual_columns = con.execute(
        'SELECT count(*) FROM information_schema.columns WHERE table_schema = ? AND table_name = ?',
        [schema, table],
    ).fetchone()[0]
    expected = (int(item['row_count']), int(item['column_count']) + 12)
    if (actual_rows, actual_columns) != expected:
        dimension_mismatches.append((schema, table, expected, (actual_rows, actual_columns)))

non_varchar_columns = con.execute("""
    SELECT count(*)
    FROM information_schema.columns
    WHERE table_schema IN ('higher_education', 'university_disclosure', 'employment')
      AND table_name <> 'safe_2023_2024_standalone'
      AND data_type <> 'VARCHAR'
""").fetchone()[0]
print({'dimension_mismatches': len(dimension_mismatches), 'non_varchar_source_columns': non_varchar_columns})
assert not dimension_mismatches
assert non_varchar_columns == 0

{'dimension_mismatches': 0, 'non_varchar_source_columns': 0}


In [4]:
legacy = con.execute("""
    SELECT count(*), min(_panel_year), max(_panel_year),
           count(*) FILTER (WHERE _panel_year IN ('2023', '2024')),
           count(*) FILTER (WHERE coalesce(개방ID, '') = '')
    FROM analysis.employment_legacy_2010_2022
""").fetchone()
standalone = con.execute("""
    SELECT count(*), min(_panel_year), max(_panel_year),
           count(*) FILTER (WHERE coalesce(학교명, '') = '')
    FROM analysis.employment_2023_2024_standalone
""").fetchone()
standalone_fields = {
    row[0] for row in con.execute("""
        SELECT column_name FROM information_schema.columns
        WHERE table_schema = 'analysis'
          AND table_name = 'employment_2023_2024_standalone'
    """).fetchall()
}
removed_relations = con.execute("""
    SELECT count(*) FROM information_schema.tables
    WHERE (table_schema = 'analysis' AND table_name = 'employment_2023_2024_resolved')
       OR (table_schema = 'employment' AND table_name = 'safe_2023_2024_resolved')
""").fetchone()[0]
print({'legacy': legacy, 'standalone': standalone, 'standalone_columns': len(standalone_fields), 'removed_legacy_relations_found': removed_relations})
assert database_summary == ('complete', 233, 180119183, 7277987, 46962, 46962)
assert legacy == (7277987, '2010', '2022', 0, 0)
assert standalone == (46962, '2023', '2024', 0)
assert '개방ID' not in standalone_fields
assert '_open_id_candidate' not in standalone_fields
assert removed_relations == 0
assert audit['validation']['panel_dimension_mismatches'] == 0
assert audit['validation']['legacy_employment_year_boundary_enforced'] is True
assert audit['validation']['standalone_employment_open_id_columns_removed'] is True
con.close()

{'legacy': (7277987, '2010', '2022', 0, 0), 'standalone': (46962, '2023', '2024', 0), 'standalone_columns': 31, 'removed_legacy_relations_found': 0}


## Takeaways

The warehouse is complete and dimensionally consistent with the panel catalog. Employment access paths are now structurally separated: use `analysis.employment_legacy_2010_2022` only for restricted OpenID longitudinal work and `analysis.employment_2023_2024_standalone` only for within-period school-name and department summaries. The removed resolved view can no longer expose inferred OpenID as a default analysis key.